# Implementing the RAG flow

The whole retrieval side, in the course's five steps:

1. chunk the text by section
2. embed every chunk
3. put the embeddings into a vector store
4. embed the user's question
5. search the store for the closest chunks

Step 6, the answer from Claude with those chunks in the prompt, is not in this lesson's code;
it is added at the end to close the loop.

The store is the course's `VectorIndex`, in [`vector_index.py`](./vector_index.py): an in-memory
list that compares the question with every vector. A real app would use a vector database;
the interface is the same (add a vector with its document, search for the k closest).

## Setup

In [1]:
from pathlib import Path

from anthropic import Anthropic
from chunking import chunk_by_section
from dotenv import load_dotenv
from embeddings import generate_embedding
from vector_index import VectorIndex

load_dotenv()
client = Anthropic()
model = "claude-haiku-4-5"

## 1. Chunk the text by section

In [2]:
chunks = chunk_by_section(Path("report.md").read_text())
print(len(chunks), "chunks")
print(chunks[2][:300], "...")  # the table of contents, as in the lesson

15 chunks
## Table of Contents

1.  Executive Summary
2.  Table of Contents
3.  Methodology
4.  Section 1: Medical Research - Understanding XDR-471 Syndrome
5.  Section 2: Software Engineering - Project Phoenix Stability Enhancements
6.  Section 3: Financial Analysis - Q3 Performance and Outlook
7.  Section 4 ...


## 2. Embed every chunk

One request for all chunks, not one per chunk. The chunks are `input_type="document"`
(the course uses the default `"query"` for everything; see the comparison below).

In [3]:
embeddings = generate_embedding(chunks, input_type="document")
print(len(embeddings), "vectors of", len(embeddings[0]), "numbers")

15 vectors of 1024 numbers


## 3. Put them into a vector store

Each vector is stored with its document: a dict with at least `content`.

In [4]:
store = VectorIndex()

for embedding, chunk in zip(embeddings, chunks, strict=True):
    store.add_vector(embedding, {"content": chunk})

print(store)

VectorIndex(count=15, dim=1024, metric='cosine', has_embedding_fn='No')


## 4. Embed the question

In [5]:
question = "What did the software engineering dept do last year?"
user_embedding = generate_embedding(question, input_type="query")

## 5. Search

Cosine **distance** = 1 - similarity: smaller is closer. `k=2` returns the two best chunks.

In [6]:
results = store.search(user_embedding, k=2)

for doc, distance in results:
    print(f"{distance:.3f}\n{doc['content'][:200]}\n")

0.623
## Executive Summary

This report synthesizes the key findings and ongoing research efforts across the organization's diverse operational and R&D departments for the past fiscal year. Our strength lie

0.648
## Section 2: Software Engineering - Project Phoenix Stability Enhancements

The Software Engineering division dedicated considerable effort to improving the stability and performance of the core syst



The **Executive Summary** beats the software section: it mentions every department, the software
work included, in one dense page. Summaries and conclusions often win this way. The answer below
still works, because the software section is the second result; with `k=1` it would be lost.
Next lessons (BM25, hybrid search, reranking) deal with exactly this.

### `query` or `document` for the chunks?

The same search with the chunks embedded the course way (`"query"`), top 3.

In [7]:
course_store = VectorIndex()
for embedding, chunk in zip(generate_embedding(chunks), chunks, strict=True):
    course_store.add_vector(embedding, {"content": chunk})

for name, index in [("document (here)", store), ("query (course)", course_store)]:
    print(name)
    for doc, distance in index.search(user_embedding, k=3):
        print(f"  {distance:.3f}  {doc['content'].splitlines()[0]}")

document (here)
  0.623  ## Executive Summary
  0.648  ## Section 2: Software Engineering - Project Phoenix Stability Enhancements
  0.741  ## Section 5: Legal Developments - Navigating IP Precedents and Regulatory Shifts
query (course)
  0.717  ## Executive Summary
  0.731  ## Section 2: Software Engineering - Project Phoenix Stability Enhancements
  0.736  ## Future Directions


The order of the top two is the same, but the gap is not. With `"document"` the third chunk is clearly
further away (0.741 against 0.648); with `"query"` all three are within 0.02 of each other, so the
cut-off between relevant and not is much less clear.

## 6. Answer with Claude

The "generation" in RAG: the found chunks go into the prompt, in XML tags, and Claude answers from them only.

In [8]:
context = "\n\n".join(doc["content"] for doc, _ in results)
prompt = f"""Answer the question using only the report excerpts below.
If the excerpts do not contain the answer, say so.

<report_excerpts>
{context}
</report_excerpts>

<question>
{question}
</question>"""

response = client.messages.create(model=model, max_tokens=1000, messages=[{"role": "user", "content": prompt}])
print("\n".join(block.text for block in response.content if block.type == "text"))
print(f"\n[{response.usage.input_tokens} input tokens; the whole report would be about {len(Path('report.md').read_text()) // 4}]")

Based on the report excerpts, the Software Engineering department did the following last year:

1. **Addressed stability issues** in the core systems of Project Phoenix, focusing on two recurring error codes:
   - `ERR_MEM_ALLOC_FAIL_0x8007000E` (memory allocation failures during peak loads)
   - `TIMEOUT_QUERY_DB_0xDEADBEEF` (data retrieval operation timeouts)

2. **Conducted root cause analysis** that identified inefficiencies in the primary data caching algorithm and suboptimal database indexing strategies

3. **Deployed a patch** for the memory allocation error, which resulted in a **40% reduction in critical failures** under simulated stress tests during Q4 2024

4. **Planned further refactoring** of the query module for the next release cycle to resolve the timeout issue

5. **Monitored system telemetry** closely for regressions or newly emerging error patterns

6. **Assisted with incident INC-2023-Q4-011** during Q4 2024

The report emphasizes that these efforts were part of imp

Next lesson: BM25 lexical search, for questions where exact words matter.